In [5]:
import dolphindb
from dolphindb._dolphindbcpp import Sink

from config import DOLPHIN

session = dolphindb.session()
_ = session.connect(
    DOLPHIN.HOST,
    DOLPHIN.PORT,
    DOLPHIN.USERNAME,
    DOLPHIN.PASSWORD,
)

In [6]:
_ = session.run("""
plugins = exec plugin from getLoadedPlugins()
if (!("MatchingEngineSimulator" in plugins)) loadPlugin("MatchingEngineSimulator")
if (!("Backtest" in plugins)) loadPlugin("Backtest")
use query
use backtest
""")

In [7]:
_ = session.run("""
coreQuerySource = table(
    timestamp([
        2025.01.02, 2025.01.02,
        2025.01.03, 2025.01.03,
        2025.01.06, 2025.01.06,
        2025.01.07, 2025.01.07
    ]) as time,
    symbol([
        "000001.SZ", "600000.SH",
        "000001.SZ", "600000.SH",
        "000001.SZ", "600000.SH",
        "000001.SZ", "600000.SH"
    ]) as code,
    double([10.0, 8.0, 11.0, 8.2, 12.0, 8.4, 13.0, 8.6]) as open,
    double([9.0, 7.5, 10.0, 7.7, 11.0, 7.9, 12.0, 8.1]) as low,
    double([11.0, 8.5, 12.0, 8.7, 13.0, 8.9, 14.0, 9.1]) as high,
    double([10.5, 8.1, 11.5, 8.3, 12.5, 8.5, 13.5, 8.7]) as close,
    long([10000, 20000, 10000, 21000, 10000, 22000, 10000, 23000]) as volume,
    double([11.0, 8.8, 12.0, 8.9, 13.0, 9.1, 14.0, 9.3]) as upLimitPrice,
    double([9.0, 7.2, 10.0, 7.4, 11.0, 7.6, 12.0, 7.8]) as downLimitPrice,
    double([9.5, 8.0, 10.5, 8.1, 11.5, 8.3, 12.5, 8.5]) as prevClosePrice,
    double([0.8, -0.2, 0.7, -0.1, 0.6, 0.1, 0.5, 0.2]) as score
)

coreDslDefinitions = fromStdJson(
    '{"selected":{"type":"DIRECT","op":"binary.gt","fields":{"left":"score","right":0.4},"params":{}}}'
)
coreDslOutputColumns = [
    "time", "code", "open", "low", "high", "close", "volume",
    "upLimitPrice", "downLimitPrice", "prevClosePrice", "score", "selected"
]
unfilteredFactorData = query::compute_factors(
    coreQuerySource,
    coreDslDefinitions
)
filteredFactorData = query::filter_factors(
    unfilteredFactorData,
    ["selected"]
)
marketData = query::project_factor_output(
    unfilteredFactorData,
    coreDslOutputColumns,
    2025.01.02,
    2025.01.08
)
msgTable = backtest::build_backtest_message(marketData)
""")

In [8]:
_ = session.run("""
config = dict(STRING, ANY)
config["startDate"] = 2025.01.02
config["endDate"] = 2025.01.07
config["strategyGroup"] = "stock"
config["cash"] = double(100000)
config["commission"] = double(0)
config["tax"] = double(0)
config["dataType"] = int(4)
config["msgAsTable"] = true
config["matchingMode"] = int(2)
config["outputOrderInfo"] = true
""")

In [9]:
class NotebookLogSink(Sink):
    def __init__(self):
        super().__init__("notebook")
        self.messages = []

    def handle(self, message):
        self.messages.append(message.log)

    def flush(self):
        pass


log_sink = NotebookLogSink()
session.msg_logger.disable_stdout_sink()
session.msg_logger.add_sink(log_sink)

try:
    _ = session.run("""
def onBar(mutable context, msg, indicator) {
    lastData = getLastData(context, msg)
    executionTime = string(context.tradeTime)
    if (lastData.rows() == 0) {
        print(
            executionTime + " 收到完整原始行情：" +
            "msgDate=" + string(date(msg.tradeTime[0])) +
            "；昨日数据为空，不下单"
        )
        return
    }

    history = getHistoryData(context, msg, false)
    print(
        executionTime + " 收到完整原始行情：" +
        "msgDate=" + string(date(msg.tradeTime[0])) +
        "；取得昨日最新截面：" +
        "lastDate=" + string(date(max(lastData.time))) +
        ", symbols=" + concat(string(lastData.code), ",") +
        ", historyLastDate=" + string(date(max(history.time)))
    )

    if (!("submitted" in context)) {
        context["submitted"] = true
        orderSymbol = symbol([
            strReplace(
                strReplace(string(lastData.code[0]), ".SZ", ".XSHE"),
                ".SH",
                ".XSHG"
            )
        ])[0]
        print(
            executionTime + " 发送订单：" +
            "市价买入 " + string(orderSymbol) +
            ", qty=100" +
            ", orderPrice=" + string(lastData.close[0])
        )
        Backtest::submitOrder(
            context.engine,
            (
                orderSymbol,
                context.tradeTime,
                0,
                lastData.close[0],
                long(100),
                1
            ),
            "minimal-buy"
        )
    }
}

def onTrade(mutable context, trade) {
    fill = trade[0]
    print(
        string(fill["tradeTime"]) + " 撮合成功：" +
        "symbol=" + string(fill["symbol"]) +
        ", price=" + string(fill["tradePrice"]) +
        ", qty=" + string(fill["tradeQty"]) +
        ", value=" + string(fill["tradeValue"]) +
        ", fee=" + string(fill["totalFee"])
    )
}

coreExampleEngine = backtest::run_backtest(
    "test-engine",
    config,
    msgTable,
    unfilteredFactorData,
    filteredFactorData,
    NULL,
    NULL,
    onBar,
    NULL,
    NULL,
    onTrade,
    NULL,
    NULL
)
stat = Backtest::getBacktestEngineStat(coreExampleEngine)
print(
    string(stat.snapshotTimestamp[0]) + " 回测结束：" +
    "status=" + string(stat.status[0]) +
    ", lastErrMsg=" + string(stat.lastErrMsg[0])
)
Backtest::dropBacktestEngine(coreExampleEngine)
""")
finally:
    session.msg_logger.remove_sink("notebook")
    session.msg_logger.enable_stdout_sink()

for message in log_sink.messages:
    print(message)

2025.01.02T15:00:00.000 收到完整原始行情：msgDate=2025.01.02；昨日数据为空，不下单
2025.01.03T15:00:00.000 收到完整原始行情：msgDate=2025.01.03；取得昨日最新截面：lastDate=2025.01.02, symbols=000001.SZ, historyLastDate=2025.01.02
2025.01.03T15:00:00.000 发送订单：市价买入 000001.XSHE, qty=100, orderPrice=10.5
2025.01.03T15:00:00.000 撮合成功：symbol=000001.XSHE, price=11, qty=100, value=1100, fee=0
2025.01.06T15:00:00.000 收到完整原始行情：msgDate=2025.01.06；取得昨日最新截面：lastDate=2025.01.03, symbols=000001.SZ, historyLastDate=2025.01.03
2025.01.07T15:00:00.000 回测结束：status=END, lastErrMsg=
